# mdprep Colab smoke test

This notebook verifies that the published `mdprep` Python distribution can be installed from PyPI in a fresh Google Colab runtime. It installs the OpenMM/PySCF extras, checks the command-line interface, runs the installed-package self-test, and validates a minimal manifest.

It deliberately does **not** claim to test a full Amber workflow: `tleap`/AmberTools, PropKa, and xTB are external executables and require a Conda-based environment.

In [ ]:
PACKAGE_SPEC = "mdprep[md,qm]==0.2.0"

import subprocess
import sys

command = [sys.executable, "-m", "pip", "install", "--upgrade", PACKAGE_SPEC]
subprocess.check_call(command)


In [ ]:
import mdprep

assert mdprep.__version__ == "0.2.0", mdprep.__version__
print(f"Installed mdprep {mdprep.__version__} from {mdprep.__file__}")
subprocess.check_call([sys.executable, "-m", "mdprep.cli", "--version"])
subprocess.check_call([sys.executable, "-m", "mdprep.cli", "selftest", "--quick"])


In [ ]:
import yaml

from mdprep.config.models import ManifestConfig

manifest = yaml.safe_load("""
project:
  name: colab_smoke
  input_structure: input.pdb
  output_dir: prepared/colab_smoke
structure:
  keep_crystal_waters: true
  remove_unknown_heterogens: false
protein:
  forcefield: ff14SB
  water_model: TIP3P
protonation:
  method: manual_only
disulfides:
  auto_detect: true
  detection_cutoff_angstrom: 2.2
  force: []
  forbid: []
ligands: []
metals: []
solvation:
  enabled: true
validation:
  fail_on_missing_parameters: true
molecular_dynamics:
  enabled: false
""")

config = ManifestConfig.model_validate(manifest)
assert config.protein.forcefield == "ff14SB"
assert config.protein.water_model == "TIP3P"
print("Minimal ff14SB/TIP3P manifest: valid")


## Expected result

The notebook finishes without an exception, reports `mdprep 0.2.0`, passes `selftest --quick`, and prints `Minimal ff14SB/TIP3P manifest: valid`.

For a full `tleap` preparation or MCPB/QM/MM calculation, use the documented Conda environment rather than this pip-only smoke test.